In [14]:
from pyspark.sql import SparkSession
import re
from pyspark.sql.types import DoubleType, IntegerType, DateType


path_raw = 'files/sales_raw.txt'
path_final_csv = 'files/sales_final.csv'


## Creates the spark session

In [3]:
spark = SparkSession.builder \
    .appName("SalesApp") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/18 17:29:58 WARN Utils: Your hostname, WIN-4JABNJNKCBT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/01/18 17:29:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/18 17:30:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## creates the empty dict

In [21]:

dict_sales = {}

## creates the empty list

In [22]:

list_sales = []

## read the lines

In [23]:

file = open(path_raw, 'r')
for line in file:
    dict_sales['transaction_id'] = re.search(r'TRANSACTION_ID: (.*?)(?=\|)', line).group(1)
    dict_sales['date'] = re.search(r'DATE: (.*?)(?=\|)', line).group(1)
    dict_sales['customer'] = re.search(r'CUSTOMER: (.*?)(?=\|)', line).group(1)
    dict_sales['product'] = re.search(r'PRODUCT: (.*?)(?=\|)', line).group(1)
    dict_sales['total'] = re.search(r'TOTAL: (.*?)(?=\|)', line).group(1)
    dict_sales['status'] = re.search(r'STATUS: (.*)', line).group(1)
    list_sales.append(dict_sales)
file.close()

In [24]:
print(list_sales)

[{'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Ergonomic_Chair ', 'total': '210.75 ', 'status': 'Pending'}, {'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Ergonomic_Chair ', 'total': '210.75 ', 'status': 'Pending'}, {'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Ergonomic_Chair ', 'total': '210.75 ', 'status': 'Pending'}, {'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Ergonomic_Chair ', 'total': '210.75 ', 'status': 'Pending'}, {'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Ergonomic_Chair ', 'total': '210.75 ', 'status': 'Pending'}, {'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Ergonomic_Chair ', 'total': '210.75 ', 'status': 'Pending'}, {'transaction_id': '1007 ', 'date': '2026-01-18 ', 'customer': 'Elena_Rius ', 'product': 'Erg

## create the spark Dataframe

In [10]:

df = spark.createDataFrame(list_sales)
df.show()

+-----------+-----------+----------------+-------+-------+--------------+
|   customer|       date|         product| status|  total|transaction_id|
+-----------+-----------+----------------+-------+-------+--------------+
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
|Elena_Rius |2026-01-18 |Ergonomic_Chair |Pending|210.75 |         1007 |
+-----------+-----------+----------------+-------+-------+--------------+



In [13]:
df.printSchema()

root
 |-- customer: string (nullable = true)
 |-- date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total: string (nullable = true)
 |-- transaction_id: string (nullable = true)



## transform data

In [15]:

df = (
    df
    .withColumn("date", df["date"].cast(DateType()))
    .withColumn("total", df["total"].cast(DoubleType()))
    .withColumn("transaction_id", df["transaction_id"].cast(IntegerType()))
    )

In [16]:
df.printSchema()

root
 |-- customer: string (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total: double (nullable = true)
 |-- transaction_id: integer (nullable = true)



## Dataframe to csv file

In [18]:

df.coalesce(1).write.csv(path_final_csv, header=True, mode="overwrite")


## Read the csv file saved before in order to check the information is correct

In [19]:

df2 = spark.read.csv(path_final_csv, header=True, inferSchema=True)
df2.show()


+----------+----------+---------------+-------+------+--------------+
|  customer|      date|        product| status| total|transaction_id|
+----------+----------+---------------+-------+------+--------------+
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
|Elena_Rius|2026-01-18|Ergonomic_Chair|Pending|210.75|          1007|
+----------+----------+---------------+-------+------+--------------+



In [20]:
df2.printSchema()

root
 |-- customer: string (nullable = true)
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total: double (nullable = true)
 |-- transaction_id: integer (nullable = true)

